# Import repaired BAFU EcoSpold files

Create or reuse a local Brightway project with **ecoinvent biosphere 3.10**, then try importing the repaired EcoSpold 1 files. Select the **bw** kernel and run through the migration and inspection cells. The optional database-write section stops while exchanges remain unresolved.

The repaired files must already exist. To generate them, run this from the repository root:

```bash
conda run --no-capture-output -n bw python "scripts/ecospold importer/repair_all.py"
```

Project storage is `artifacts/brightway/` (ignored by Git). The first project setup downloads Brightway's biosphere archive and requires internet access.

In [1]:
import os
import sys
from pathlib import Path


ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "scripts/ecospold importer").is_dir()
)
SOURCE = ROOT / "data/processed/ecospold1-schema-fixed"
STORAGE = ROOT / "artifacts/brightway"

PROJECT = "bafu-2026-biosphere-310"
BIOSPHERE = "ecoinvent-3.10-biosphere"
DATABASE = "BAFU:2026"

# Configure storage and the local helper path before importing Brightway.
STORAGE.mkdir(parents=True, exist_ok=True)
os.environ["BRIGHTWAY2_DIR"] = str(STORAGE)
sys.path.insert(0, str(ROOT / "scripts/ecospold importer"))

In [2]:
# Run the setup cell above first.
import bw2data as bd
import bw2io as bi
from date_compat import xml_date_parser
from timestamp_compat import iso_timestamp_parser

/opt/homebrew/Caskroom/miniforge/base/envs/bw/lib/python3.11/site-packages/scikits/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)


15:46:40+0200 [info     ] Using environment variable BRIGHTWAY2_DIR for data directory:
/Users/romain/GitHub/lca-data-lineage-hackathon/artifacts/brightway


## Create the project

Reuse the same project as the Python import script, or create it from Brightway's biosphere 3.10 archive on first use.

In [3]:
if PROJECT not in bd.projects:
    bi.install_project("ecoinvent-3.10-biosphere", project_name=PROJECT)
    
bd.projects.set_current(PROJECT)

In [4]:
bd.databases

Databases dictionary with 1 object(s):
	ecoinvent-3.10-biosphere

## Extract the repaired files

Use the local date/timestamp adapters and the standard EcoSpold 1 importer strategies. The XML files remain unchanged. The `importer` object stays available for inspection.

In [5]:
with iso_timestamp_parser(), xml_date_parser():
    importer = bi.SingleOutputEcospold1Importer(str(SOURCE), DATABASE, use_mp=False)

importer.apply_strategies()

  1%|▋                                                                            | 106/11947 [00:00<00:23, 497.43it/s]/opt/homebrew/Caskroom/miniforge/base/envs/bw/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:408: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
100%|███████████████████████████████████████████████████████████████████████████| 11947/11947 [00:24<00:00, 479.91it/s]


Extracted 11947 datasets in 24.92 seconds
Applying strategy: normalize_units
Applying strategy: assign_only_product_as_production
Applying strategy: clean_integer_codes
Applying strategy: drop_unspecified_subcategories
Applying strategy: strip_biosphere_exc_locations
Applying strategy: update_ecoinvent_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: link_technosphere_by_activity_hash
Applied 9 strategies in 0.98 seconds


## Apply the technosphere migrations

Edit the [general mappings](../schemas/mappings/bafu-2026-technosphere.json) or the [source-file-specific mappings](../schemas/mappings/bafu-2026-technosphere-context.json). The helper reads both files on every run, registers them in the current project, applies the corrections, and reruns technosphere linking.

The second file distinguishes identical exchange labels used by different consuming datasets. Its source-file marker is temporary and is removed after migration. See the [mapping notes](../schemas/mappings/README.md) for evidence and documented assumptions.

After changing either JSON file, rerun the extraction cell above and the following cells. Raw and repaired XML files remain unchanged.

In [6]:
from technosphere_migrations import apply_technosphere_migrations

apply_technosphere_migrations(importer, ROOT / "schemas/mappings")

Applying strategy: migrate_datasets
Applying strategy: migrate_exchanges
Applied 24 rules from bafu-2026-technosphere.json
Applying strategy: migrate_datasets
Applying strategy: migrate_exchanges
Applied 10 rules from bafu-2026-technosphere-context.json
Applying strategy: link_iterable_by_fields


In [7]:
importer.match_database(
    fields=["name", "reference product", "location"], edge_kinds=["technosphere"]
)

Applying strategy: link_iterable_by_fields


## Normalize biosphere categories and link

Apply the [category migration](../schemas/mappings/bafu-2026-biosphere-categories.json) before matching against biosphere 3.10. This translates BAFU compartment labels while keeping flow names, units, amounts, uncertainty, and comments unchanged. Matching uses the complete **name, categories, and unit**.

Unresolved flows remain in `importer.data`. The helper writes their occurrence counts, example source files, and diagnostic groups to `reports/generated/biosphere-unlinked-after-categories.json`. See the [biosphere migration notes](../docs/bafu-2026-biosphere-migrations.md). Rerun extraction and all migration cells after editing the mappings.

Category normalization is the first pass; the remaining name, unit, and compartment mismatches still require review before a complete inventory can be written.

In [8]:
from biosphere_migrations import apply_biosphere_category_migration

biosphere_result = apply_biosphere_category_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-categories.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-categories.json",
)
biosphere_result

Applying strategy: migrate_datasets
Applying strategy: migrate_exchanges
Applying strategy: link_iterable_by_fields
Biosphere: 170,259 linked; 123,488 unlinked across 1,342 signatures.
Unlinked biosphere report: /Users/romain/GitHub/lca-data-lineage-hackathon/reports/generated/biosphere-unlinked-after-categories.json


{'before': {'total': 293747,
  'linked': 0,
  'unlinked': 293747,
  'unlinked_signatures': 2679},
 'after': {'total': 293747,
  'linked': 170259,
  'unlinked': 123488,
  'unlinked_signatures': 1342}}

## Apply reviewed biosphere names and units

The [flow migration](../schemas/mappings/bafu-2026-biosphere-flows.json) handles explicit chemical/name aliases, equivalent particulate and biogenic labels, redundant `/m3` suffixes, and exact Bq → kBq or kWh → MJ conversions. Compartments stay unchanged. bw2io applies the documented conversion factors to the amounts and their uncertainty parameters.

Every rule must match exactly one target flow in biosphere 3.10. This stage's unresolved-flow report is `reports/generated/biosphere-unlinked-after-flows.json`; the category-only result is saved separately. See the [evidence and limitations](../docs/bafu-2026-biosphere-flow-migrations.md). Rerun extraction and all migration cells after editing a mapping.

In [9]:
from biosphere_migrations import apply_biosphere_flow_migration

biosphere_flow_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-flows.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-flows.json",
)
biosphere_flow_result

Applying strategy: migrate_datasets
Applying strategy: migrate_exchanges
Applying strategy: link_iterable_by_fields
Biosphere: 200,822 linked; 92,925 unlinked across 1,003 signatures.
Unlinked biosphere report: /Users/romain/GitHub/lca-data-lineage-hackathon/reports/generated/biosphere-unlinked-after-flows.json


{'before': {'total': 293747,
  'linked': 170259,
  'unlinked': 123488,
  'unlinked_signatures': 1342},
 'after': {'total': 293747,
  'linked': 200822,
  'unlinked': 92925,
  'unlinked_signatures': 1003}}

## Apply historical biosphere catalog mappings

Use the [catalog migration](../schemas/mappings/bafu-2026-biosphere-catalog.json) to follow documented ecoinvent UUID correspondences, catalog synonyms, and historical renames. Source CAS values are included in matching, so an explicitly identified chromium(VI) exchange is handled separately from the generic Chromium label. Amounts, units, compartments, and original CAS metadata are retained.

The [catalog evidence notes](../docs/bafu-2026-biosphere-catalog-migrations.md) describe the source catalogs and confidence limits.

In [10]:
biosphere_catalog_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-catalog.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-catalog.json",
)
biosphere_catalog_result

Applying strategy: migrate_datasets
Applying strategy: migrate_exchanges
Applying strategy: link_iterable_by_fields
Biosphere: 269,140 linked; 24,607 unlinked across 637 signatures.
Unlinked biosphere report: /Users/romain/GitHub/lca-data-lineage-hackathon/reports/generated/biosphere-unlinked-after-catalog.json


{'before': {'total': 293747,
  'linked': 200822,
  'unlinked': 92925,
  'unlinked_signatures': 1003},
 'after': {'total': 293747,
  'linked': 269140,
  'unlinked': 24607,
  'unlinked_signatures': 637}}

## Map documented resource names and water units

Apply the [resource migration](../schemas/mappings/bafu-2026-biosphere-resources.json). Legacy ore-composition names are replaced by element names according to ecoinvent's documented 3.10 migration. Water emissions expressed in kilograms are converted to the target catalog's cubic-metre convention (1,000 kg per m³), including their uncertainty parameters. Raw XML remains unchanged.

See the [resource evidence notes](../docs/bafu-2026-biosphere-resource-migrations.md). This step retains every exchange and writes `reports/generated/biosphere-unlinked-after-resources.json`.

In [11]:
biosphere_resource_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-resources.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-resources.json",
)
biosphere_resource_result

Applying strategy: migrate_datasets
Applying strategy: migrate_exchanges
Applying strategy: link_iterable_by_fields
Biosphere: 274,939 linked; 18,808 unlinked across 585 signatures.
Unlinked biosphere report: /Users/romain/GitHub/lca-data-lineage-hackathon/reports/generated/biosphere-unlinked-after-resources.json


{'before': {'total': 293747,
  'linked': 269140,
  'unlinked': 24607,
  'unlinked_signatures': 637},
 'after': {'total': 293747,
  'linked': 274939,
  'unlinked': 18808,
  'unlinked_signatures': 585}}

## Apply additional reviewed biosphere mappings

Apply the [reviewed migration](../schemas/mappings/bafu-2026-biosphere-reviewed.json): chemical spelling/synonym corrections, omitted resource subcategories, and the approved regional-water and land-class mappings. Regional emissions become generic `Water`; regional withdrawals retain their water type (river, lake, cooling, or unspecified origin), with the original name, unit, categories, and region retained in each exchange's `bafu original biosphere` metadata. Generic LCIA factors do not use that retained region automatically. Water kg → m³ uses the documented factor 0.001, including uncertainty rescaling.

The approved broader industrial-area and intensive-forest classes also retain their original detailed labels in `bafu original biosphere` metadata. Their linked LCIA factors use the broader class. Unsupported flows remain unlinked. See the [evidence and limits](../docs/bafu-2026-biosphere-reviewed-migrations.md). This stage writes `reports/generated/biosphere-unlinked-after-reviewed.json`.

In [12]:
biosphere_reviewed_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-reviewed.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-reviewed.json",
)

Applying strategy: migrate_datasets
Applying strategy: migrate_exchanges
Applying strategy: link_iterable_by_fields
Biosphere: 286,125 linked; 7,622 unlinked across 277 signatures.
Unlinked biosphere report: /Users/romain/GitHub/lca-data-lineage-hackathon/reports/generated/biosphere-unlinked-after-reviewed.json


## Correct reviewed water and resource contexts

Apply the [water/context migration](../schemas/mappings/bafu-2026-biosphere-water-context.json). It resolves well-water and fossil-water resource names, corrects explicitly identified resource compartments, and covers the remaining supported regional water labels. Fossil-water emissions use the existing `water / fossil well` target. Process-water withdrawal labels retain their stated groundwater category where present; other unspecified-origin withdrawals use the general water-resource class.

All rules retain their original name, unit, and categories on the exchange, with the region where stated. Water kg → m³ uses the documented 0.001 factor. See the [evidence and scope](../docs/bafu-2026-biosphere-water-context-migrations.md).

In [13]:
biosphere_water_context_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-water-context.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-water-context.json",
)

Applying strategy: migrate_datasets
Applying strategy: migrate_exchanges
Applying strategy: link_iterable_by_fields
Biosphere: 287,170 linked; 6,577 unlinked across 251 signatures.
Unlinked biosphere report: /Users/romain/GitHub/lca-data-lineage-hackathon/reports/generated/biosphere-unlinked-after-water-context.json


## Apply reviewed chemical names and elemental mass

Apply the [elemental-mass migration](../schemas/mappings/bafu-2026-biosphere-chemistry.json): TiO₂ → titanium, KCl → potassium, and barite (BaSO₄) → barium. The approved factors represent the contained element's mass, so both source and target units are kilograms while their material basis differs. bw2io rescales the amount and uncertainty; the helper checks the factor against the stated formula and atomic weights.

Each exchange retains its original compound label, unit, and categories in `bafu original biosphere`, plus the formula, target element, and atomic weights in `bafu elemental conversion`. See the [evidence and factors](../docs/bafu-2026-biosphere-chemistry-migrations.md).

Five further catalog/chemical aliases cover VOC of unspecified origin, dimethylformamide, DSMA, TCMTB, and Tin (II). Their full compartments, units, and amounts are retained; original names remain on the exchanges.

In [14]:
biosphere_chemistry_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-chemistry.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-chemistry.json",
)

Applying strategy: migrate_datasets
Applying strategy: migrate_exchanges
Applying strategy: link_iterable_by_fields
Biosphere: 287,711 linked; 6,036 unlinked across 242 signatures.
Unlinked biosphere report: /Users/romain/GitHub/lca-data-lineage-hackathon/reports/generated/biosphere-unlinked-after-chemistry.json


## Apply the approved resource-gas unit assumption

The [gas-unit migration](../schemas/mappings/bafu-2026-biosphere-gas-units.json) maps natural-gas and mine-gas resources from `cubic meter` or `normal cubic meter` to their unique `standard cubic meter` targets. This is the approved **1:1 label assumption**: source reference conditions remain unknown, and no temperature/pressure conversion is performed.

Original names, categories, and units remain in `bafu original biosphere`; `bafu unit assumption` records the decision. No bw2io multiplier is used, so amounts and all uncertainty fields stay exactly unchanged. See the [scope and verification](../docs/bafu-2026-biosphere-gas-unit-migrations.md). This stage writes `reports/generated/biosphere-unlinked-after-gas-units.json`.

In [15]:
biosphere_gas_unit_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-gas-units.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-gas-units.json",
)
biosphere_gas_unit_result

Applying strategy: migrate_datasets
Applying strategy: migrate_exchanges
Applying strategy: link_iterable_by_fields
Biosphere: 288,074 linked; 5,673 unlinked across 238 signatures.
Unlinked biosphere report: /Users/romain/GitHub/lca-data-lineage-hackathon/reports/generated/biosphere-unlinked.json


{'before': {'total': 293747,
  'linked': 287711,
  'unlinked': 6036,
  'unlinked_signatures': 242},
 'after': {'total': 293747,
  'linked': 288074,
  'unlinked': 5673,
  'unlinked_signatures': 238}}

## Apply documented land and resource mappings

The [land/resource migration](../schemas/mappings/bafu-2026-biosphere-land-resources.json) follows official historical renames for seabed infrastructure, non-irrigated crop classes, and graphite. It also applies the approved broader classes for tropical rainforest and organic cropland/pasture, corrects the peat resource compartment, completes three approved regional-water aliases, and links a helium resource whose source dataset explicitly describes extraction from natural gas.

All incoming labels remain in `bafu original biosphere`, including Europe for the water rows. Amounts, units, and uncertainty are unchanged. The broader land targets use generic LCIA factors; retained tropical/organic detail is not automatically characterized. See the [evidence and validation](../docs/bafu-2026-biosphere-land-resource-migrations.md). This stage writes `reports/generated/biosphere-unlinked-after-land-resources.json`.

In [ ]:
biosphere_land_resource_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-land-resources.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-land-resources.json",
)
biosphere_land_resource_result

## Apply the reviewed follow-up mappings

The [follow-up migration](../schemas/mappings/bafu-2026-biosphere-followup.json) applies the approved broader unspecified-land classes and generic Swiss rail-land targets. Original labels and categories remain in `bafu original biosphere`; rail rows also retain `region: CH`. Generic LCIA does not automatically use these retained distinctions.

It also corrects Cesium-136 to Caesium-136 with an exact Bq → kBq conversion (×0.001), classifies the reviewed limestone/petroleum CO₂ exchange as fossil, and links three exact 2-chlorophenol synonyms in water. Every other amount and uncertainty field stays unchanged. See the [evidence and limitations](../docs/bafu-2026-biosphere-followup-migrations.md). This stage writes `reports/generated/biosphere-unlinked-after-followup.json`.

In [ ]:
biosphere_followup_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-followup.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-followup.json",
)
biosphere_followup_result

## Apply reviewed compartment mappings

The [compartment migration](../schemas/mappings/bafu-2026-biosphere-compartments.json) moves the two landfill-mass/carbon indicators from `natural resource / in ground` to their official `inventory indicator / waste` category. Their names, amounts, signs, units, and uncertainty stay unchanged; original categories are retained in `bafu original biosphere`.

These existing biosphere targets are ecological-scarcity inventory indicators.

The same stage applies the approved generic-air targets for 913 high-altitude emissions, retaining `air / lower stratosphere + upper troposphere` in audit metadata. Amounts and uncertainty stay unchanged; generic LCIA will no longer distinguish their altitude. It also applies the approved broader compartments for 14 urban-air/surface-water/industrial-soil emissions, 75 suspended-solids emissions to fossil water, and 38 indoor TCDD emissions. Five further surface-water exchanges use the same approved generic-water approach after checking their exact chemical aliases. Every original compartment remains in audit metadata. Generic factors do not automatically represent the retained indoor exposure or aquifer distinction.

See the [definitions and verification](../docs/bafu-2026-biosphere-compartment-migrations.md). The final unresolved report is `reports/generated/biosphere-unlinked.json`.

In [ ]:
biosphere_compartment_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-compartments.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked.json",
)
biosphere_compartment_result

In [ ]:
importer.statistics()

In [ ]:
importer.data[0]

In [ ]:
for u in list(importer.unlinked)[:10]:
    if u["type"] == "biosphere":
        print(u)

## Optional database write and LCA

Keep unsupported flows unresolved pending supported targets. The check below stops **Run All** before the existing drop/write/LCA cells while any exchange remains unlinked. The migration and diagnostic cells above retain all inventory rows. Do not run the later `drop_unlinked` cell while following this preservation workflow.

In [ ]:
unlinked_count = sum(1 for _ in importer.unlinked)
if unlinked_count:
    raise RuntimeError(
        f"{unlinked_count:,} exchanges remain unresolved. "
        "Review reports/generated/biosphere-unlinked.json and retain these exchanges; "
        "database writing is deferred until supported targets are established."
    )

In [ ]:
importer.drop_unlinked(i_am_reckless=True)

In [ ]:
importer.write_database()

In [ ]:
import bw2calc as bc
method = bd.methods.random()
act = bd.Database("BAFU:2026").random()
lca = bc.LCA({act: 1}, method)
lca.lci()
lca.lcia()
print(lca.score)

In [ ]:
act.as_dict()